In [117]:
import os 
import pickle 
import time 
import streamlit as st 
from dotenv import load_dotenv 
from langchain_text_splitters import RecursiveCharacterTextSplitter 
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import UnstructuredURLLoader 
from langchain_community.vectorstores import FAISS 

# This automatically loads OPENAI_API_KEY into the system environment
load_dotenv() 

True

In [118]:
llm = ChatGroq(
    model="groq/compound",
    temperature=0.4, 
    max_tokens=600
)
# as prompt is gievn out of context window of llm 
# from langchain_community.document_loaders import AsyncHtmlLoader
# from langchain_community.document_transformers import Html2TextTransformer


# urls=[
#     "https://docs.langchain.com"
# ]
# loader = AsyncHtmlLoader(urls)
# raw_html = loader.load()
# transformer = Html2TextTransformer()
# data = transformer.transform_documents(raw_html)
# len(data)

loader = UnstructuredURLLoader(urls=[
    "https://www.pinecone.io/learn/vector-database/"
])

data = loader.load()
data[0].page_content


"← Learn\n\nWhat is a Vector Database & How Does it Work? Use Cases + Examples\n\nRoie Schwaber-Cohen\n\nRoie Schwaber-Cohen\n\nMay 3, 2023Core Components\n\nShare:\n\nJump to section:\n\nWhat is a Vector Database?\n\nServerless Vector Databases\n\nSummary\n\nWhat is a Vector Database?\n\nA vector database indexes and stores vector embeddings for fast retrieval and similarity search, with capabilities like CRUD operations, metadata filtering, horizontal scaling, and serverless.\n\nWe’re in the midst of the AI revolution. It’s upending any industry it touches, promising great innovations - but it also introduces new challenges. Efficient data processing has become more crucial than ever for applications that involve large language models, generative AI, and semantic search.\n\nAll of these new applications rely on vector embeddings, a type of vector data representation that carries within it semantic information that’s critical for the AI to gain understanding and maintain a long-term m

**CHUNKING**

In [119]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1100,
    chunk_overlap = 200
)

docs = text_splitter.split_documents(data)
len(docs)


33

**EMBEDDING**

In [120]:
embed = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vec_idx_openai = FAISS.from_documents(docs,embed)

file_path = "vec_db.pkl"

with open(file_path,'wb') as f : 
    pickle.dump(vec_idx_openai,f)

Loading weights: 100%|█████████████████████████████████████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 6429.92it/s]


In [121]:
if os.path.exists(file_path):
    with open(file_path,"rb") as f:
        vec_idx = pickle.load(f)

vec_idx

**RETRIVAL**

In [122]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate



In [128]:
retriever = vec_idx.as_retriever(search_kwargs={"k": 3})

def format_docs(docs):
    formatted = []
    for doc in docs:
        source = doc.metadata.get("source","UNKNOWN URL")
        val = f"Source : {source}\nContent : {doc.page_content}"
        formatted.append(val)
        
    return "\n\n".join(formatted)
    
context = format_docs(docs)
context

"Source : https://www.pinecone.io/learn/vector-database/\nContent : ← Learn\n\nWhat is a Vector Database & How Does it Work? Use Cases + Examples\n\nRoie Schwaber-Cohen\n\nRoie Schwaber-Cohen\n\nMay 3, 2023Core Components\n\nShare:\n\nJump to section:\n\nWhat is a Vector Database?\n\nServerless Vector Databases\n\nSummary\n\nWhat is a Vector Database?\n\nA vector database indexes and stores vector embeddings for fast retrieval and similarity search, with capabilities like CRUD operations, metadata filtering, horizontal scaling, and serverless.\n\nWe’re in the midst of the AI revolution. It’s upending any industry it touches, promising great innovations - but it also introduces new challenges. Efficient data processing has become more crucial than ever for applications that involve large language models, generative AI, and semantic search.\n\nAll of these new applications rely on vector embeddings, a type of vector data representation that carries within it semantic information that’s c

In [129]:
template = """You are an expert documentation assistant. Use the following extracted context to answer the question. 
If you do not know the answer based on the context, say exactly "I cannot find that in the provided documents." Do not invent facts.

Context:
{context}

Question: {question}
Answer:"""

prompt = ChatPromptTemplate.from_template(template)

In [130]:
lcel_chain = (
    {
    "context" : retriever | format_docs,
        "question" : RunnablePassthrough()
        
}

  | prompt
  | llm
  | StrOutputParser()
    
)
print("LCEL chain setup complete")

LCEL chain setup complete


In [131]:
def ask(ques):
    query = ques

    print("Analyzing documentation chunks...")
    answer = lcel_chain.invoke(query)
    
    print("\n🤖 Assistant Answer:")
    print("-" * 50)
    print(answer)
    print("-" * 50)
    

In [132]:
ques = 'how to to effectively manage and maintain a vector database?'
ask(ques)

Analyzing documentation chunks...

🤖 Assistant Answer:
--------------------------------------------------
**How to effectively manage and maintain a vector database**

Based on the provided Pinecone documentation, the key practices for managing and maintaining a vector database are:

1. **Leverage built‑in data‑management operations**  
   - Use the database’s native **insert**, **delete**, and **update** APIs.  
   - These operations are straightforward and eliminate the extra integration work required when using a standalone index such as FAISS.

2. **Store and use metadata for fine‑grained control**  
   - Attach descriptive **metadata** to each vector record.  
   - Query vectors not only by similarity but also by metadata filters (e.g., date, category, user ID) to narrow results and improve relevance.

3. **Plan for scalability from the start**  
   - Choose a vector database that **scales horizontally** (distributed and parallel processing) and can handle growing data volumes and

**other ways to avoid context window overflow and geeting error 413**

1.Strategy 1: The Character/Token Cutoff (The Quick Fix)The easiest way to guarantee you never crash is to intercept the text right before it hits the LLM and chop off anything that exceeds your limit.You can modify your format_docs function to act as a gatekeeper:

In [134]:
def format_docs(docs):
    formatted = []
    total_chars = 0
    MAX_CHARS = 8000  # Safety ceiling (roughly 2,000 tokens)

    for doc in docs:
        source = doc.metadata.get("source", "UNKNOWN URL")
        chunk_text = f"Source: {source}\nContent: {doc.page_content}\n\n"
        
        # Check if adding this chunk pushes us over the limit
        if total_chars + len(chunk_text) > MAX_CHARS:
            break  # Stop adding chunks to protect the LLM window
            
        formatted.append(chunk_text)
        total_chars += len(chunk_text)
        
    return "".join(formatted)

2.Strategy 2: Switch to "Map-Reduce" Architecture (The Structural Fix)Instead of stuffing all retrieved chunks into a single prompt (which LangChain calls the Stuff method), you split the work into two steps: Map and Reduce.Map: Send each chunk to the LLM individually and ask it to extract only the specific facts relevant to the user's question. (e.g., if you have 5 chunks, you make 5 ultra-small API calls).Reduce: Take those 5 tiny summaries, combine them, and send them to the LLM for the final answer.This completely bypasses context windows because the LLM never looks at all the raw documents at the same time.

3. Strategy 3: The Context-Aware Re-Ranker (The Smart Fix)Instead of letting FAISS guess which 5 chunks are best based purely on math, you use a Re-ranker (like Cohere or BGE-Reranker).You tell your retriever to pull a large amount of data (e.g., k=20).The Re-ranker reads those 20 chunks and filters them down to the absolute 2 or 3 best sentences, discarding the rest.This compresses the information density so you only send high-quality data to Groq.
